CI/CD PER MO DELLI AI (GITHUB)

Finora abbiamo addestrato modelli sul nostro pc. MA nel mondo reale non possiamo fare il deployment a mano, è troppo pericoloso.
Vediamo come usare la GitHub Actiont per creare un catena di montaggio automattizzata che testa, valida e spedisce il codice nel cloud in totale sicurezza.

L'obbiettivo è costruire un'architettura di rilascio solida

Per iniziare, non serve un software esetrno. GitHub Actions è integrato nel repository. 
Tutto parte da una cartella magica da creare nella root del progetto (.github/workflows) e inserire un file .yml
Questo file è come il manuale di istruzioni per GitHub.
Appena carichi questo file, GitHub rileva la configurazione e farà apparire una tab dedicata chiamata 'Actions' nel menu superiore del vostro repository, dove potrete monitorare ogni esecuzione.

CI/CD è il processo che controlla una modifica al codice e la porta verso il rilascio. In un progetto (a cui lavorano più colleghi) è quello che può succedere tra il momento in cui invii una modifica a  un .py su Git e il momento in cui i colleghi usano la nuova versione. La sequenza automatica di questi passaggi di chiama pipeline
CI (continuous integration): integri spesso piccole modifiche nel codice condiviso. La pipeline installa le dipendenze ed esegui i test su ogni versione che le chiedi di controllare.
CD (continuous delivery): dopi i controlli, la versione è pronta per il rilascio. Puoi averla già installata in collaudo e decidere quindo pubblicarla sul server che usano i colleghi.
CD (continuous deployment): ogni modifica destinata al rilascio che supera i controlli previsti viene pubblicata automaticamente in produzione.

Dopo aver modificato il codice del file .py e dopo aver fatto git push, puoi configurare uno strumento come GitHub Actions per avviare i controlli su quella versione.
Un test chiede a Flask GET/ e verifica che la pagina si apra. Se capita per esempio 'TemplateNotFound' il test fallisce e la pipeline ferma il rilascio.
Un altro test chiede GET /giacenza/ABC1234 e verifica che il JSON riporti 120. Così controlli anche il risultato di una ricerca, oltre all'apertura della pagina.
Praticamente puoi configurare GitHub Actions perchè esegua determinati comandi a ogni git push (esempio1: http://120.0.0.61:5000/?articolo=ABC1234 esempio2:http://120.0.0.61:5000, ecc )
Ogni volta configurato e partito il GitHub Actions è il test a inviare get/ e verificare che Flask risponda 
Se il progetto restituisce dati statici, il valore restituito sarà sempre uguale ed al test è possibile passare direttamente il valore atteso.
Discorso diverso se collego l'app al database ERP, in questo caso la giacenza di magazzino può cambiare ed il valore mandato al test essere sbagliato. In questo caso è necessario utilizzare un database di collaudo, in modo che i risultati siano sempre uguali.
Il punto fondamentale è tenere ripetibili i dati della prova. Se altri utenti fanno movimenti nel collaudo, perpara i dati di ART_TEST all'inizio di ogni esecuzione.

Ma come facciamo a dire a GitHub: ei ora devi inziare a lavorare?

Definizione dei Trigger
Decidere quando far scattare i test.
- Evento on-push: nel file YAML, usate la chiave 'on push' per avviare i test ogni volta che caricate codice sui branch 'main' o 'develop'
- Pull Request Trigger: fondamentale per i team, la pipeline si attiva quando qualcuno propone una modifica, garantendo che il codice sia testato prima di entrare nel branch principale.

Ma cosa deve controllare GitHub dentro questi test?
solo se il codice gira? No, non basta, dobbiamo testare l'intera architettura.
Usiamo Pytest per le funzioni base, come la pulizia dei testi. Ma dobbiamo andare oltre.
Implementiamo i Tensor Shape Checks, se cambiate un layer della rete e la forma del tensore non è più la stessa, il sistema crash in produzione, Github deve accorgersene subito.

Scaricare le libreria AI è lentissimo, come evitiamo di perdere ore ad ogni test?
GitHub mette a disposizione l'azione 'actions/cache' per memorizzare i pacchetti installati tra una run e l'altra.
L'efficienza temporale del workflow T si riduce drasticamente grazie all'uso di chiavi di cache basate sull'hash dei file dei requisiti.
Se le librerie sono le stesse le recupera istantaneamente.

Superati i test del codice validiamo l'intelligenza del modello.
qui la CI tradizonale diventa mlops
Non vogliamo che un modello con accuratezza insufficiente venga distribuito. Creeremo un Gate logico basato sulla valutazione automatica.
Dobbiamo creare un evaluation, lo script di validazione deve restituire un 'Exit Code 1' se le performance sul Golden Set non superano la soglia minima impostata dal team.
GitHub inerpreta questo sengale come un fallimento e blocca la strada verso il rilascio.

Configurazione del Gate
Impostazioni di sicurezza nel repository
- Branc Protection Rules: andate nel menu 'Settings' del repository cliccate su 'Branches' e aggiungere una regola per il branch 'main'
- Require Status Checks: attivare l'opzione 'Requires status checks to pass before merging' e cercare il nome del vostro job di valutazione definito nello YAML
- Blocking Merge: una volta attivata, il tasto 'Merge' su GitHub diventa grigio e inattivo se la valutazione dell'AI non ha dato esito positivo.
- L'autorizzazione al merge è una variabile booleana dipendente dal superamento della soglia di accuratezza 'theta'
E' una protezione matematica, non si entra in produzione se la soglia theta non è superata.

Ma per superare i testi il codice ha bisogno delle chiavi.
Dove le mettiamo?
Gestione dei Segreti
- Menu Secrets: per scasricare i pesi o i dati da cloud (S3/Azuere), andate in Setting - Secrets and variables - Actions - e cliccate su 'New Repository secret'
- Variabili d'ambiente: Nello YAML, richiamate queste chiavi tramite la sintassi '${{secrets.NOME_CHIAVE}}' per usarle negli scirpt senza espoerle nel codice
- Artifacts Storage: usate 'actions/upload-artifact' per salvare i report PDF o i grafici prodotti dalla valutazione, scaricabili direttamente dalla pagina della Action conclusa.

E se il test richiede una GPU?
Selezione dei Runner
Eseguire test su hardware specifico
Se la valutazione del modello richiede una GPU, GitHub permette di configurare 'Self-hosted runners'. Dovete andare in 'settings -actions - runners' e seguire la guida per registrare il vostro server.
Puoi collegare il vostro server locale o un istanza cloud a github, la pipeline usa la vostra potenza di calcolo
La velocità di inferenza del test dipende dal throughtput del runner selezionato nel campo 'runs-on'

Una volta che il modello è validato, il prodotto va spedito
Il codice è salvo, il modello è intelligente.

Continous Delivery (CD)
Automatizzare il Rilascio
L'ultimo miglio consiste nel trasformare il codice verificato in un'immagine Docket pronta per la produzione. GitHub Actions gestisce l'intero processo di build e push
Utilizziamo GitHub Container Registry (ghcr.io) come deposito sicuro, accessibile direttamente tramite i permessi nativi del vostro account.

Packaging e Pubblicazione
Creare l'artefatto finale
- Login al Registry: utilizzate l'azione 'docker/login-action' fornendo 'GITHUB_TOKEN' generato automaticamente da GitHub per ogni sessione.
- Build and Push: configurate lo step per compilare il Dockerfile e caricarlo con un tag univoco, solitamente l'hash del commit Git
- Deployment Trigger: La CD deve scattare solo se tutti i test della CI sono terminati con successo, usando la clausola 'needs' nello YAML
- Il tempo di deploy totale è la somma del tempo di build del container e della latenza di caricamente nel registry cloud.

Tracciabilità del Rilascio
- Tagging Semantico: assegnate alle immagini tag come 'latest' per l'uso corrente e tag specifici come 'v1.2.0' basati sulle 'GitHub Releases' per storicizzare i rilasci.
- Multi-stage Build: spiegazione verbale: Nel Dockerfile, separate l'ambiente di build da quello finale per ottenere immagini più leggere e sicure, prive di compilatori o codici sorgenti inutili.
- Slack/Teams Notifications: configurate un 'Webhook' nel menu 'settings' di GitHub per inviare un messaggio automatico al team (su slack o microsoft team) quando il nuovo modello è online.

E se il nuovo modello si comporta in modo strano una volta online?
La relisienza è la capacità di tornare indietro.
Rollback Automatico.
Gestione degli imprevisti
GitHub permette di definire workflos di roolback. Se il deploy fallisce, una Actions specifica può ripubblicare l'immagine Docker precedente identificata dal tag hash salvato.
La resilienza del sistema è data dalla capacità di ripristinare lo stato stabile precedente in tempi minimi.



